In [69]:
from dotenv import load_dotenv
import os
import requests
from typing import *
import sys
import subprocess
import shlex
from datetime import datetime
import json
import uuid
import pandas as pd
from google.cloud import storage

# Add path to import custom modules
# sys.path.append(os.path.abspath("../src"))

load_dotenv()

True

In [71]:
# logger_config.py
import logging

def configure_logging():
    # Configure the root logger
    root_logger = logging.getLogger()
    
    # Check if handlers already exist to avoid duplicates
    if root_logger.handlers:
        return
        
    # Create a stream handler for console output
    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)
    
    # Create formatter and add to the handler
    formatter = logging.Formatter('[%(asctime)s] %(name)s:%(lineno)d - %(levelname)s - %(message)s')
    console_handler.setFormatter(formatter)
    
    # Add the handler to the root logger
    root_logger.addHandler(console_handler)
    root_logger.setLevel(logging.INFO)

def get_module_logger(module_name):
    # Make sure logging is configured
    configure_logging()
    
    # Return a logger with the module name
    return logging.getLogger(module_name)

In [72]:
# Auxilary functions
def partition_id_by_year_quarter(p):
    return "".join(p.get('display_name').split(" ")[:2])

def partition_id_by_year(p):
    return p.get('display_name').split(" ")[0]

def no_of_parts_in_partition(p):
    return int(p.get('display_name').replace("(", "").replace(")","").split(" ")[-1])

def part_size_mb(p):
    return float(p.get('size_mb'))

def get_total_size(json):
    return round(sum([p.get('size_mb') for p in json.get('partitions')]),2)

def read_json_file(json_path):
    with open(json_path, "r") as f:
        d = json.load(f)
        return d
        
def get_filename(url):
    return ".".join("_".join(url.split("/")[-2:]).split(".")[:-1])

def download_file(url, download_path="tmp"):
    filename = get_filename(url)
    os.makedirs(download_path,exist_ok=True)
    subprocess.run(
        f'wget -q -O - {url} | gunzip > {os.path.join(download_path,filename)}',
        shell=True,
    )

def filter_partition(years='', partitions=[]):
    years = [y.strip() for y in years.split(",")]
    if not years:
        print("No args provided")
        return
    return [p for p in partitions if p.get('partition_id') in years]

In [73]:
# Extract_drug_events
"""
Function to restructure JSON object to handle batch processing better
"""
def extract_drug_events(json):
    """Restructures JSON object to handle batch processing better"""
    if not json or 'results' not in json:
        raise ValueError("Invalid input: Missing 'results' key.")

    events = json.get('results',{}).get('drug',{}).get('event',{})
    total_records = events.get('total_records')
    partitions = events.get('partitions',[])

    # Generate unique partition_id and its count
    partition_ids = {}
    for p in partitions:
        # Extract year as partition_id
        id = partition_id_by_year(p)

        # Number of occurences
        partition_ids[id] = partition_ids.get(id,0) + 1
    
    # Groups partition by partitionid
    results = []
    for item in partition_ids.items():
        id, count = item
        file_list = []
        counter = 0
        tot_size = 0
        records = 0

        for p in partitions:
            if counter == count:
                break
            if partition_id_by_year(p) == id:
                counter+=1
                file_list.append(p.get('file'))
                tot_size+=part_size_mb(p)
                records+=p.get('records')

        results.append(
            {
                "partition_id": id,
                "records" : records,
                "count": count,
                "size_mb" : round(tot_size,2),
                "files" : file_list
            }
        )
    
    return {
        "total_records" : total_records,
        "partitions" : results
    }

In [74]:
# Create Batch
"""
Function to seggregate partitions as batches based on disksize threshold
"""
def create_batch(partitions, max_batch_size_mb=10000):
    batch = []                  # partitions per batch
    batch_partitions = []       # Partitions under the threshold
    big_batch_partitions = []   # Different approach to process bigger partitions
    sum_size = 0                # Size counter

    for p in partitions:
        size = p.get('size_mb', 0)

        if size > max_batch_size_mb:
            # TODO:
            # Handle oversized partititions
            big_batch_partitions.append(p)
            continue
        
        if sum_size + size > max_batch_size_mb:
            # TODO:
            # - Declare batch_partitions as batch #
            # - Reset sum_size
            # - Reset batch_partitions
            batch.append(batch_partitions.copy())
            batch_partitions.clear()
            sum_size = 0
            continue
        
        batch_partitions.append(p)
        sum_size += size

    # Flush batch_partitions to schedule as last batch
    if len(batch_partitions) != 0:
        batch.append(batch_partitions.copy())
        batch_partitions.clear()
    
    return batch, big_batch_partitions

In [75]:
"""Adverse Drug Event"""
# import os
# import uuid
# import pandas as pd
# from utilities import get_module_logger
import hashlib

logger = get_module_logger(__name__)

class ADE:
    # Patient information
    patient_header = [
        "patientid",
        "recordyear",
        "patientagegroup",
        "patientonsetage",
        "patientonsetageunit",
        "patientsex",
        "patientweight",
        "fulfillexpeditecriteria",                           
        "primarysourcecountry",                              
        "occurcountry",                                      
        "reporttype",                                        
        "receiptdate",
        "receivedate",
        "safetyreportid",
        "transmissiondate",                                  
        "serious",
        "seriousnesscongenitalanomali",                      
        "seriousnessdeath",
        "seriousnesshospitalization",
        "seriousnessdisabling",
        "seriousnesslifethreatening",
        "seriousnessother",
    ]

    # Drug information
    drug_header = [
        "patientid",
        "recordyear",
        "actiondrug",                                     
        "drugcharacterization",                           
        "medicinalproduct",
        "activesubstancename",
        "drugindication",    
        "drugadministrationroute",    
        "drugstartdate",
        "drugenddate",
        "drugdosagetext",
        "drugstructuredosagenumb",
        "drugstructuredosageunit",
        "drugtreatmentduration",
        "drugtreatmentdurationunit",
        "drugrecurreadministration",
    ]

    # Reaction information
    reaction_header = [
        "patientid",
        "recordyear",
        "reactionmeddrapt",
        "reactionoutcome",
    ]

    def __init__(self, year):
        self.year = year
        self.patients_list = []
        self.drugs_list = []
        self.reactions_list = []
    
    def extractJSON(self, json):
        data = json.get('results')
        for item in data:
            patientid = str(uuid.uuid4())
            patient = item.get("patient",{})

            self.patients_list.append((
                patientid,
                self.year,
                patient.get("patientagegroup"),
                patient.get("patientonsetage"),
                patient.get("patientonsetageunit"),
                patient.get("patientsex"),
                patient.get("patientweight"),
                item.get("fulfillexpeditecriteria"),                            # Added
                item.get("primarysourcecountry"),                               # Added
                item.get("occurcountry"),                                       # Added
                item.get("reporttype"),                                         # Added
                item.get("receiptdate"),
                item.get("receivedate"),
                item.get("safetyreportid"),
                item.get("transmissiondate"),                                   # Added
                item.get("serious"),
                item.get("seriousnesscongenitalanomali"),                       # Added
                item.get("seriousnessdeath"),
                item.get("seriousnesshospitalization"),
                item.get("seriousnessdisabling"),
                item.get("seriousnesslifethreatening"),
                item.get("seriousnessother"),
            ))

            drugs = patient.get('drug',[])
            for drug in drugs:
                self.drugs_list.append((
                    patientid,
                    self.year,
                    drug.get("actiondrug"),                                     # Added
                    drug.get("drugcharacterization"),                           # Added
                    

                    drug.get("medicinalproduct"),
                    drug.get("activesubstance",{}).get("activesubstancename"),
                    drug.get("drugindication"),    
                    drug.get("drugadministrationroute"),    
                    drug.get("drugstartdate"),
                    drug.get("drugenddate"),
                    drug.get("drugdosagetext"),
                    drug.get("drugstructuredosagenumb"),
                    drug.get("drugstructuredosageunit"),
                    drug.get("drugtreatmentduration"),
                    drug.get("drugtreatmentdurationunit"),
                    drug.get("drugrecurreadministration"),
                ))

            reactions = patient.get("reaction",[])
            for reaction in reactions:
                self.reactions_list.append((
                    patientid,
                    self.year,
                    reaction.get("reactionmeddrapt"),
                    reaction.get("reactionoutcome"),
                ))
    def row_count(self,):
        df_p, df_d, df_r = self._to_dataframe()
        return df_p.shape[0], df_d.shape[0], df_r.shape[0]
                
    def _row_hash(self, df, cols_to_hash):
        df_subset = df[cols_to_hash].fillna("null").astype(str).apply(lambda col: col.str.lower())
        concatenated = df_subset.agg('|'.join, axis=1)
        df['row_hash'] = [hashlib.sha256(s.encode()).hexdigest() for s in concatenated]
        return df
    
    def _content_hash(self, df):
        row_hashes = df['row_hash'].sort_values().to_list()
        compound = "".join(row_hashes)
        return hashlib.sha256(compound.encode()).hexdigest()
    
    def get_hash(self):
        df_patient, df_drug, df_reaction  = self._to_dataframe(row_hash=True)
        patient_hash = self._content_hash(df_patient)
        drug_hash = self._content_hash(df_drug)
        reaction_hash = self._content_hash(df_reaction)

        return patient_hash, drug_hash, reaction_hash

    def _to_dataframe(self, row_hash = False):
        df_patient = pd.DataFrame(self.patients_list, columns=self.patient_header)
        df_drug = pd.DataFrame(self.drugs_list, columns=self.drug_header)
        df_reaction = pd.DataFrame(self.reactions_list, columns=self.reaction_header)

        if row_hash:
            df_patient = self._row_hash(df_patient, sorted(self.patient_header[1:]))
            df_drug = self._row_hash(df_drug, sorted(self.drug_header[1:]) )
            df_reaction = self._row_hash(df_reaction, sorted(self.reaction_header[1:]))
        
        return df_patient, df_drug, df_reaction

    def save_as_parquet(self, save_to, fname):
        df_patients, df_drugs, df_reactions = self._to_dataframe(row_hash=True)
        df = [df_patients, df_drugs, df_reactions]

        dirs = []

        for p in ["patient", "drug", "reaction"]:
            path = os.path.join(save_to, p, self.year)
            dirs.append(path)
            if not os.path.exists(path):
                logger.info(f"Directory '{path}' missing. Created '{path}'")
                os.makedirs(path, exist_ok=True)
        
        for d,p in zip(df, dirs):
            saved_path = os.path.join(p,f"{fname}.parquet")
            d.to_parquet(saved_path)
            logger.info(f"Parquet File saved to: {saved_path}")

In [76]:
# from datetime import datetime, timezone
# year = 2004
# records = 12000
# schema = ['patient','drug','reaction`']
# metadata = {
#     s : {
#         'year' : year,
#         'total_records' : records,
#         'files' : {},
#     }
#     for s in schema
# }

# metadata['patient']

In [77]:
# metadata['patient']['files'].setdefault('file1.parquet',{})['records'] = 10000 
# metadata['patient']['files'].setdefault('file1.parquet',{})['hash'] = "test" 

In [78]:
# metadata['patient']

In [79]:
# os.makedirs("./temp/pq/patient/2004",exist_ok=True)
# with open("./temp/pq/patient/2004/_METADATA.json", "w") as f:
#     json.dump(metadata['patient'], f, indent=4)

In [80]:
# for s in schema:
#     metadata[s]['generated_at'] = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

In [81]:
# # Download partition's files
# for i in partition[0].get('files'):
#     download_file(i)

In [82]:
def upload_to_gcs(local_base_dir, bucket_name, gcs_prefix):
    """Uploads files from local directory to a GCS bucket."""
    
    creds_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
    if not creds_path:
        raise EnvironmentError("GOOGLE_APPLICATION_CREDENTIALS not set in .env or environment.")

    client = storage.Client()
    bucket = client.bucket(bucket_name)

    for root, _, files in os.walk(local_base_dir):
        for file in sorted(files):
            local_file_path = os.path.join(root, file)
            relative_path = os.path.relpath(local_file_path, local_base_dir)
            gcs_blob_path = os.path.join(gcs_prefix, relative_path).replace("\\", "/")
            blob = bucket.blob(gcs_blob_path)
            blob.upload_from_filename(local_file_path)
            
            logger.info(f"Uploaded {local_file_path} to gs://{bucket_name}/{gcs_blob_path}")

In [83]:
# gcs_prefix = "data/pq"
# local_base_dir = "./tmp"
# for item in os.walk(local_base_dir):
#     # Scans each level of the path and lists 
#     root, dir, files = item
#     print(item)
#     # print(f"Root: {root}")
#     # print(f"Directory: {dir}")
#     # print(f"Files: {sorted(files)}")
#     for f in files:
#         # if f.endswith(".ipynb"):
#         local_file_path = os.path.join(root,f) # Path relative to the parent folder. 'root' here is basically the parent folder of f
#         print(f"local_file_path: {local_file_path}")
#         relative_path = os.path.relpath(local_file_path, ".") # If there are deeper levels in the root directory then this gives us the path from root to this file
#         print(f"relative_path: {relative_path}")
#         gcs_path = os.path.join(gcs_prefix, relative_path)
#         print(f"gcs_path: {gcs_path}")


In [138]:
def fetch_metadata_from_gcs(schema, year, bucket):
    client = storage.Client()
    bucket = client.bucket(bucket)
    metadata = {}
    logger.info("Fetching metadata")
    for s in schema:
        blob = bucket.get_blob(f"data/pq/{s}/{year}/_METADATA.json")
        if blob is not None:
            metadata[s] = json.loads(blob.download_as_text())
        else:
            logger.info(f"Blob not found")
            return {}
    
    return metadata

def validate_hash(ade, metadata, filename):
    """
    returns
        `True` : Records and Hash match
        `False` : Records and Hash mismatch
    """
    logger.info("Validating Hash...")
    if len(metadata.keys()) < 1:
        return False
    
    for s, count, hash in zip(metadata.keys(), ade.row_count(), ade.get_hash()):
        metadata[s]['files'].setdefault(f'{filename}.parquet', {})
        file_metadata = metadata[s]['files'][f'{filename}.parquet']
        file_metadata.setdefault('content_hash','')
        file_metadata.setdefault('records',-1)

        logger.info(f"Schema: {s}")
        logger.info(f'source hash: {hash}')
        logger.info(f"blob hash: {file_metadata['content_hash']}")

        if (file_metadata['records'] != count) or (file_metadata['content_hash'] != hash):
            return False
    
    return True


In [139]:
from datetime import datetime, timezone
def process_batch(batch, metrics, bucket):

    logger.info("Initiating Batch Processing")

    # Create temp directories
    TEMP_DIR = "./temp/"
    RAW_DIR = os.path.join(TEMP_DIR,"raw")
    PQ_DIR = os.path.join(TEMP_DIR,"pq")
    tmp_dirs = [RAW_DIR, PQ_DIR]

    if not os.path.exists(RAW_DIR):
        logger.info("Directory 'raw' missing. Created 'raw'")
        os.makedirs(RAW_DIR,exist_ok=True)

    # Batch iteration
    for i, b in enumerate(batch):
        logger.info('===================================================================')
        logger.info(f'============================= BATCH {i+1} =============================')
        logger.info('===================================================================')

        # metrics.reset()

        # Partitioon iteration
        for j,p in enumerate(b):
            logger.info(f'----------------- Processing partition {j+1} -----------------')  

            schema = ['patient', 'drug', 'reaction']
            year = p.get('partition_id')
            files = p.get('files')      
            total_count = p.get('count')
            file_count = 1

            """
            TODO
            - Check if blob exists in the bucket
                -> Yes
                    -> Fetch metdata
                    -> Check if metadata changed
                        -> Yes
                            -> Update metadata
                            -> Reprocess file
                        -> No
                            -> Skip upload
                -> No (DONE)
                    -> Create metadata
                    -> Process and save Parquet
                    -> Save metadata

            """

            # Fetch Metadata
            # IF NOT EXISTS: Create Metadata for each of the schemas for the same year
            # IF EXISTS: Initialize `metadata` with the existing one
            metadata = fetch_metadata_from_gcs(schema, year, bucket)

            # URL iteration
            for f in files:
                filename = f"drug-event-part-{file_count}-of-{total_count}"

                try:
                    logger.info(f"Download started: {f}")
                    dl_filepath = os.path.join(RAW_DIR,f"{filename}.json")  
                    subprocess.run(
                        f'wget -q -O - {shlex.quote(f)} | gunzip > {shlex.quote(dl_filepath)}',
                        shell=True,
                        check=True,
                        capture_output=True,
                        text=True
                    )

                    # Saved to tmp folder ./temp/raw/drug-event-part-1-of-x.json
                    logger.info(f"File saved to: {dl_filepath}")

                    # Load JSON and map to class ADE
                    ade = ADE(year)
                    temp_json = read_json_file(dl_filepath)
                    ade.extractJSON(temp_json)

                    # metrics.update(ade)

                    logger.info(f"Parsed json file to ADE object: {dl_filepath}")

                    # Check against existing hash for each schema from metadata
                    # If hash != meta_hash: perform save_as_parquet, update meta_hash
                    # Else skip save, move to next file in partition
                    if validate_hash(ade, metadata, filename):
                        logging.info("Hash and Count validated. No Changes Required.")
                        file_count+=1
                        continue

                    # Initialize/Update metadata for each schema
                    for s, hash, count in zip(schema, ade.get_hash(), ade.row_count()):
                        metadata.setdefault(s, {})
                        metadata[s]['year'] = year
                        metadata[s]['total_records'] = metadata[s].get('total_records',0) + count 

                        metadata[s].setdefault('files',{})
                        metadata[s]['files'].setdefault(f'{filename}.parquet',{})
                        metadata[s]['files'][f'{filename}.parquet']['records'] = count
                        metadata[s]['files'][f'{filename}.parquet']['content_hash'] = hash

                    # Save as parquet file to ./temp/pq/<schema>/<year>/drug-event-part-1-of-x.parquet
                    ade.save_as_parquet(save_to=PQ_DIR, fname=filename)
                    file_count+=1

                except subprocess.CalledProcessError as e:
                    logger.error(f"(return {e.returncode}) Failed to download or unzip: {f}")
                    logger.error(f"{e.stderr.strip()}")
                except Exception as e:
                    logger.error(f"Unexpected error occured: {e}")

            # Need a flag here to trigger update
            for s in schema:
                # Update timestamp
                metadata[s]['generated_at'] = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
                meta_path = os.path.join(PQ_DIR, s, year)
                os.makedirs(meta_path, exist_ok=True)
                logging.info(f"Metadata path: {meta_path}")
                with open(os.path.join(meta_path,"_METADATA.json"), "w") as f:
                    json.dump(metadata[s], f, indent=4)
            

            # Upload metadata and parquet files to GCS bucket
            upload_to_gcs(local_base_dir=PQ_DIR, bucket_name=bucket, gcs_prefix="data/pq")
            logger.info(f"Uploaded partition '{year}' parquet files to GCS.")

            # Purge tmp folder to prepare for next partition
            for dir_path in tmp_dirs:
                logger.info(f"Purging files in'{dir_path}'")
                wildcard_path = os.path.join(dir_path, "*")
                popen = subprocess.Popen(f"rm -rfv {wildcard_path}", stdout=subprocess.PIPE, shell=True, text=True)

                for o in popen.stdout:
                    logger.info(o.strip())
            
            logger.info("Purge completed")

        logger.info('===================================================================')
        logger.info(f'============================= Batch {i+1} END =========================')
        logger.info('===================================================================')
    
        # Publish Metrics here
        # metrics.publish()

    # Clear temp folder here
    logger.info("Deleting temporary directories")
    popen = subprocess.Popen(f"rm -rvf {TEMP_DIR}", stdout=subprocess.PIPE, shell=True,text=True)
    for o in popen.stdout:
        logger.info(o.strip())

    logger.info("Batch Processing Completed!")

In [140]:
# SCHEMA = ['patient', 'drug', 'reaction']
# metadata = {}

# for s in SCHEMA:
#     metadata.setdefault(s,{})
#     metadata[s]['year'] = 2004
#     metadata[s]['total_records'] = metadata[s].get('total_records',0) + 10

#     metadata[s].setdefault('files',{})
#     metadata[s]['files'].setdefault('filename', {})
#     metadata[s]['files']['filename']['records'] = 10
#     metadata[s]['files']['filename']['hash'] = 'hash1'

# pprint(metadata, sort_dicts=False)

In [141]:
# from pprint import pprint
# BUCKET = "ade-pipeline-bucket"
# SCHEMA = ['patient', 'drug', 'reaction']
# year = 2004

# def fetch_metadata_from_gcs(schema, year, bucket):
#     client = storage.Client()
#     bucket = client.bucket(bucket)
#     metadata = {}

#     for s in schema:
#         blob = bucket.get_blob(f"data/pq/{s}/{year}/_METADATA.json")
#         if blob is not None:
#             metadata[s] = json.loads(blob.download_as_text())
#         else:
#             print(f"Blob not found")
#             return {}
    
#     return metadata

# metadata = fetch_metadata_from_gcs(SCHEMA, 2004, BUCKET)
# metadata

In [142]:
URL = "https://api.fda.gov/download.json"
MAX_BATCH_SIZE_MB = 13000
PROMETHEUS_GATEWAY = None
JOB = "openfda_ingestion"
BUCKET = "ade-pipeline-bucket"

year = '2004'

res = requests.get(URL)
data = res.json()
downloads_json = extract_drug_events(data)
partitions = downloads_json.get('partitions')
filtered_parititons = filter_partition(year, partitions) 
batch, _ = create_batch(filtered_parititons,max_batch_size_mb=MAX_BATCH_SIZE_MB)
batch

[[{'partition_id': '2004',
   'records': 214812,
   'count': 20,
   'size_mb': 1029.58,
   'files': ['https://download.open.fda.gov/drug/event/2004q3/drug-event-0001-of-0005.json.zip',
    'https://download.open.fda.gov/drug/event/2004q3/drug-event-0002-of-0005.json.zip',
    'https://download.open.fda.gov/drug/event/2004q3/drug-event-0003-of-0005.json.zip',
    'https://download.open.fda.gov/drug/event/2004q3/drug-event-0004-of-0005.json.zip',
    'https://download.open.fda.gov/drug/event/2004q3/drug-event-0005-of-0005.json.zip',
    'https://download.open.fda.gov/drug/event/2004q2/drug-event-0001-of-0005.json.zip',
    'https://download.open.fda.gov/drug/event/2004q2/drug-event-0002-of-0005.json.zip',
    'https://download.open.fda.gov/drug/event/2004q2/drug-event-0003-of-0005.json.zip',
    'https://download.open.fda.gov/drug/event/2004q2/drug-event-0004-of-0005.json.zip',
    'https://download.open.fda.gov/drug/event/2004q2/drug-event-0005-of-0005.json.zip',
    'https://download.o

In [144]:
from time import time
start = time()
process_batch(batch, None, BUCKET)
end = time()

print(f"Finished in {end-start:.4f} seconds")

[2025-06-09 23:29:32,699] __main__:4 - INFO - Initiating Batch Processing
[2025-06-09 23:29:32,700] __main__:13 - INFO - Directory 'raw' missing. Created 'raw'
[2025-06-09 23:29:32,701] __main__:18 - INFO - ===================================================================
[2025-06-09 23:29:32,702] __main__:19 - INFO - ============================= BATCH 1 =============================
[2025-06-09 23:29:32,702] __main__:20 - INFO - ===================================================================
[2025-06-09 23:29:32,703] __main__:26 - INFO - ----------------- Processing partition 1 -----------------
[2025-06-09 23:29:32,758] __main__:5 - INFO - Fetching metadata
[2025-06-09 23:29:33,739] __main__:62 - INFO - Download started: https://download.open.fda.gov/drug/event/2004q3/drug-event-0001-of-0005.json.zip
[2025-06-09 23:29:39,666] __main__:73 - INFO - File saved to: ./temp/raw/drug-event-part-1-of-20.json
[2025-06-09 23:29:41,580] __main__:82 - INFO - Parsed json file to ADE object

Finished in 116.3839 seconds


Finished in 162.6597 seconds

In [ ]:
# for f in os.listdir('tmp'):
#     filepath = os.path.join('tmp',f)
#     file_json = read_json_file(filepath)
    
#     # ade = ADE(year=2004)
#     # ade.extractJSON(file_json)
#     # break

### Benchmark - row operation vs vectorized

In [ ]:
# import hashlib
# import time

# # Not vectorized
# start = time.time()
# # .apply(axis=1) performs function calls on each row (across columns) which is slower
# test_hash = patient_df[cols_to_hash].fillna("null").astype('str').apply(
#     lambda row: hashlib.sha256("|".join(row.str.lower()).encode()).hexdigest(),
#     axis=1
#     )
# end = time.time()
# print(f"Original apply row-wise took {end - start:.4f} seconds")
# test_hash.head()

Original apply row-wise took 0.6515 seconds


0    1511b146ec4d7679484a2269dead15bd6b2bb12faefcd7...
1    73b940db21c7eae03487b1ee0ec0c8737292b272fd8ca5...
2    4ed0e926e062b153d44d9589ef22654cfc3671f326bb6a...
3    a68ac4330991a95fbeb5583724b3011655a66c6555025b...
4    a401372c18978c556a704316f077b8b0c6257b782f244e...
dtype: object

In [ ]:
# # Vectorized and faster
# start = time.time()

# df_sub = patient_df[cols_to_hash].fillna("null").astype(str).apply(lambda col: col.str.lower())
# concatenated = df_sub.agg('|'.join, axis=1)
# patient_df['row_hash'] = [hashlib.sha256(s.encode()).hexdigest() for s in concatenated]
# end = time.time()
# print(f"Vectorized string ops + list hashing took {end - start:.4f} seconds")

Vectorized string ops + list hashing took 0.1090 seconds
